# EcoVision AI — Step 3: YOLO11s Baseline Training
**Intelligent Waste Detection & Smart Segregation System**
* **Model:** Ultralytics YOLO11s (`yolo11s.pt` - COCO pretrained weights)
* **Classes (6):** `plastic`, `metal`, `cardboard`, `glass`, `paper`, `general_trash`
* **Dataset:** Curated TACO (2,543 images, 8,119 annotations, leakage-free stratified 70/15/15 split)
* **Hardware:** Google Colab GPU (NVIDIA T4 / V100 / A100)
* **Reproducibility Seed:** 42

> **Important Working Rules:**
> 1. We establish an honest, reproducible baseline without fabricating metrics.
> 2. Validation is run strictly on the `val` split; the `test` split remains **SEALED** for final evaluation.
> 3. Checkpoints and evaluation metrics are persisted automatically to Google Drive.


## 1. Verify GPU Acceleration
Ensure Colab runtime is set to GPU: **Runtime > Change runtime type > T4 GPU**.


In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise SystemError("GPU not detected! Please enable GPU in Runtime > Change runtime type.")


## 2. Mount Google Drive
Mount Google Drive to persist weights (`best.pt`, `last.pt`), training curves, and logs across session timeouts.


In [ ]:
from google.colab import drive
import os
import shutil
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/EcoVisionAI')
DRIVE_WEIGHTS = DRIVE_DIR / 'weights'
DRIVE_RESULTS = DRIVE_DIR / 'results'

DRIVE_WEIGHTS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"[OK] Google Drive mounted. Persistence directory: {DRIVE_DIR}")


## 3. Install Dependencies
Install Ultralytics YOLO11, PyYAML, Pillow, and Roboflow.


In [ ]:
!pip install -q "ultralytics>=8.3.0" pyyaml pillow tqdm

import ultralytics
from ultralytics import YOLO
print(f"[OK] Ultralytics version: {ultralytics.__version__}")


## 4. Setup Dataset in Colab
Choose **Option A** (upload zip from local machine) OR **Option B** (extract from Google Drive) OR **Option C** (download and run verified pipeline).

**Option A / B (Fastest — Recommended):**
Zip your local `data/processed` folder (`ecovision_processed_dataset.zip`) and place it in your Google Drive folder `MyDrive/EcoVisionAI/`, then run the block below.


In [ ]:
import zipfile
import shutil

DATA_DIR = Path('/content/data/processed')

# Check if zip is on Google Drive
drive_zip = DRIVE_DIR / 'ecovision_processed_dataset.zip'
local_zip = Path('/content/ecovision_processed_dataset.zip')

if drive_zip.exists():
    print(f"Found dataset zip on Google Drive: {drive_zip}")
    shutil.copy(drive_zip, local_zip)
elif not local_zip.exists():
    print("Please upload 'ecovision_processed_dataset.zip' directly to Colab (/content/) or Google Drive (MyDrive/EcoVisionAI/)")

if local_zip.exists():
    print("Extracting dataset...")
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/data')
    print(f"[OK] Dataset extracted to {DATA_DIR}")
else:
    print("[WARN] No pre-packaged zip found. Please upload dataset or see Option C below.")


## 5. Generate Colab-Specific Dataset YAML
Write a dataset YAML with absolute Colab paths pointing directly to `/content/data/processed`.


In [ ]:
colab_yaml_content = """# EcoVision AI - Colab Dataset Configuration
path: /content/data/processed
train: images/train
val: images/val
test: images/test

nc: 6
names:
  0: plastic
  1: metal
  2: cardboard
  3: glass
  4: paper
  5: general_trash
"""

COLAB_YAML_PATH = Path('/content/dataset_colab.yaml')
COLAB_YAML_PATH.write_text(colab_yaml_content.strip())
print(f"[OK] Created {COLAB_YAML_PATH}")

# Verify file counts
for split in ['train', 'val', 'test']:
    imgs = list(Path(f'/content/data/processed/images/{split}').glob('*.*'))
    lbls = list(Path(f'/content/data/processed/labels/{split}').glob('*.txt'))
    print(f"Split '{split}': {len(imgs)} images, {len(lbls)} labels")
assert len(list(Path('/content/data/processed/images/train').glob('*.*'))) > 0, "Train split is empty! Please check dataset extraction."


## 6. Initialize Pretrained YOLO11s
We load `yolo11s.pt` (small variant, ~9.4M parameters).
* Pretrained on COCO (80 classes)
* Transferred to our 6 waste categories via fine-tuning
* License: AGPL-3.0


In [ ]:
model = YOLO('yolo11s.pt')
print("Model Architecture & Pretrained Weights Loaded successfully:")
print(f"Parameters: ~9.4M")
print(f"Task: Object Detection (detect)")


## 7. Train Baseline Model
* **Epochs:** 50
* **Batch Size:** 16 (automatically reduced if GPU memory is constrained)
* **Image Size:** 640x640
* **Optimizer:** AdamW (initial lr = 0.001)
* **Patience:** 15 epochs (Early Stopping)
* **Save Period:** Every 5 epochs
* **Seed:** 42 (Reproducible)
* **Deterministic:** True


In [ ]:
TRAIN_CONFIG = {
    'data': str(COLAB_YAML_PATH),
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'patience': 15,
    'save': True,
    'save_period': 5,
    'device': 0,
    'workers': 2,
    'project': '/content/runs/train',
    'name': 'yolo11s_waste_baseline',
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'seed': 42,
    'deterministic': True,
    'exist_ok': True
}

# Run training
try:
    results = model.train(**TRAIN_CONFIG)
    print("[OK] Training completed successfully!")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("[WARN] CUDA Out of Memory! Retrying with batch size 8...")
        torch.cuda.empty_cache()
        TRAIN_CONFIG['batch'] = 8
        results = model.train(**TRAIN_CONFIG)
    else:
        raise e


## 8. Persist Checkpoints and Curves to Google Drive
Copy `best.pt`, `last.pt`, `results.csv`, confusion matrix, and training curves to Drive so no results are lost.


In [ ]:
RUN_DIR = Path('/content/runs/train/yolo11s_waste_baseline')

# Copy best and last weights
if (RUN_DIR / 'weights' / 'best.pt').exists():
    shutil.copy2(RUN_DIR / 'weights' / 'best.pt', DRIVE_WEIGHTS / 'best.pt')
    shutil.copy2(RUN_DIR / 'weights' / 'last.pt', DRIVE_WEIGHTS / 'last.pt')
    print(f"[OK] Checkpoints saved to Google Drive: {DRIVE_WEIGHTS}")

# Copy results CSV, curves, confusion matrix
for f in RUN_DIR.glob('*.*'):
    if f.is_file():
        shutil.copy2(f, DRIVE_RESULTS / f.name)
print(f"[OK] Training curves & logs saved to Google Drive: {DRIVE_RESULTS}")


## 9. Comprehensive Validation Evaluation
Evaluate the trained `best.pt` model strictly on the **validation split (`val`)**.
We report:
* Precision (P), Recall (R)
* mAP@50, mAP@50-95
* Per-class metrics
* Confusion matrix
*(The test split remains sealed and is NOT evaluated here).*


In [ ]:
# Load best model
best_model = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))

# Validate on val split
val_results = best_model.val(
    data=str(COLAB_YAML_PATH),
    split='val',
    batch=16,
    imgsz=640,
    conf=0.25,
    iou=0.6,
    plots=True,
    device=0
)

print("\n" + "=" * 60)
print("  BASELINE VALIDATION METRICS (YOLO11s)")
print("=" * 60)
print(f"Overall mAP@50     : {val_results.box.map50:.4f}")
print(f"Overall mAP@50-95  : {val_results.box.map:.4f}")
print(f"Overall Precision  : {val_results.box.p.mean():.4f}")
print(f"Overall Recall     : {val_results.box.r.mean():.4f}")
print(f"Overall F1-Score   : {val_results.box.f1.mean():.4f}")

print("\nPer-Class Breakdown:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'mAP@50':<12} {'mAP@50-95':<12}")
print("-" * 63)
for i, name in enumerate(['plastic', 'metal', 'cardboard', 'glass', 'paper', 'general_trash']):
    p = val_results.box.p[i] if i < len(val_results.box.p) else 0.0
    r = val_results.box.r[i] if i < len(val_results.box.r) else 0.0
    map50 = val_results.box.maps[i] if i < len(val_results.box.maps) else 0.0
    print(f"{name:<15} {p:<12.4f} {r:<12.4f} {map50:<12.4f}")


## 10. Visualize Training Curves & Confusion Matrix
Render the generated confusion matrix, PR curve, and loss curves directly in the notebook.


In [ ]:
from IPython.display import Image, display

plots = [
    RUN_DIR / 'results.png',
    RUN_DIR / 'confusion_matrix.png',
    RUN_DIR / 'PR_curve.png',
    RUN_DIR / 'F1_curve.png'
]

for p in plots:
    if p.exists():
        print(f"\n--- {p.name} ---")
        display(Image(filename=str(p)))


## 11. Error Analysis: Inspect Sample Predictions
Run inference on validation images to inspect genuine detections, false positives, and missed items.


In [ ]:
import glob
import random

val_images = list(Path('/content/data/processed/images/val').glob('*.*'))
sample_images = random.sample(val_images, min(5, len(val_images)))

pred_results = best_model.predict(
    source=sample_images,
    conf=0.25,
    save=True,
    project='/content/runs/predict',
    name='val_samples'
)

print("Sample predictions saved to /content/runs/predict/val_samples")
for p in list(Path('/content/runs/predict/val_samples').glob('*.jpg'))[:3]:
    display(Image(filename=str(p)))
